In [56]:
import numpy as np
import pandas as pd
from scipy.io import loadmat
import matplotlib.pyplot as plt
import pandas as pd
import time
import os
import sys
import zarr
import napari 
import dask.array as da
import math 

## Input variables

Refer to the  ISS- Correlate segmented endocytic spots to cell location (apical, basal, lateral) in 3D for more detailed infomation

In [71]:
# Change this variable to where your python packages are. Don't remove the r
package_dir = r'C:\Users\Kenny\LLSM-CME-ANALYSIS\Final\src'

# Change this variable to the directory where all your data is. Don't remove the r
base_dir = r'C:\Users\Kenny\akamatsu'

# Change this variable to the current data you are analyzing in your base_dir.
input_file_directory = 'lumenoid_1full_analysis/'

# Change this variable to the center of the lumenoid is. Input of it is a list[0,1,2] -> [x,y,z]
lumen_center = [167,205,91]

# Change this variable to the apical point that is closer to the center. Input of it is a list[0,1,2] -> [x,y,z]
apical_start_point = [109,195,83]

# Change this variable to the apical point that is farther to the center. Input of it is a list[0,1,2] -> [x,y,z]
apical_end_point = [127,148,78]

# Change this variable to the basal point that is closer to the center. Input of it is a list[0,1,2] -> [x,y,z]
basal_start_point = [151,32,82]

# Change this variable to the basal point that is farther to the center. Input of it is a list[0,1,2] -> [x,y,z]
basal_end_point = [165,14,106]


## Preliminary code for setup

In [72]:
pythonPackagePath = os.path.abspath(package_dir)
sys.path.append(pythonPackagePath)

from intensity_time_plots import filter_track_ids_by_length_ranges, random_track_ids
from intensity_time_plots import intensity_time_plot, createBufferForLifetimeCohort
from intensity_time_plots import createBufferForLifetimeCohort_normalized, cumulative_plots, cumulative_plots_ax

zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)

input_directory_tracks = 'datasets'
input_directory_filtered = 'filtered_tracks_final.pkl'
input_directory_all_tracks_full = os.path.join(base_dir, input_file_directory) + 'datasets/track_df_cleaned_final_full.pkl'

input_directory_full_filtered_tracks = (os.path.join(base_dir, input_file_directory, 
                                                    input_directory_tracks, input_directory_filtered))

def find_radius(point):
    x1 = lumen_center[0]
    x2 = point[0]

    y1 = lumen_center[1]
    y2 = point[1]

    z1 = lumen_center[2]
    z2 = point[2]
    answer = math.sqrt((x2-x1) ** 2 + (y2-y1) ** 2 + (z2-z1) ** 2) 
    # answer = math.sqrt((x2-x1) ** 2 + (y2-y1) ** 2)
    return answer

apical_start_edge = find_radius(apical_start_point)
apical_end_edge = find_radius(apical_end_point)
basal_start_edge = find_radius(basal_start_point)
basal_end_edge = find_radius(basal_end_point)
print(apical_start_edge)

59.39696961966999


In [85]:
track_df = pd.read_pickle(input_directory_all_tracks_full)
filtered_tracks = pd.read_pickle(input_directory_full_filtered_tracks)

membrane_regions = []
track_radius = []
for index, row in filtered_tracks.iterrows():
    test_point = [row[0].iloc[0], row[1].iloc[0], row[2].iloc[0]]
   # test_point = [row[0].iloc[0], row[1].iloc[0]]
    test_radius = find_radius(test_point)
        
    # Check if z_value falls within basal ranges
    if basal_start_edge <= test_radius < basal_end_edge:
        membrane_regions.append('Basal')

    # Check if z_value falls within apical ranges
    elif apical_start_edge <= test_radius < apical_end_edge:
        membrane_regions.append('Apical')
        
    # elif test_radius < apical_start_edge or test_radius > basal_end_edge:
    #     print('Not in the lumenoid', index)
    #     print(test_radius)
    #     print(apical_start_edge)
    #     print(basal_end_edge)
    #     print(row)
    else:
        membrane_regions.append('Lateral')
    track_radius.append(test_radius)

filtered_tracks['membrane_region'] = membrane_regions
filtered_tracks['track_radius'] = track_radius
print(apical_start_edge)
print(basal_start_edge)

59.39696961966999
173.97126199461795


C:\Users\Kenny\AppData\Local\Temp\ipykernel_4128\54858982.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  test_point = [row[0].iloc[0], row[1].iloc[0], row[2].iloc[0]]


In [86]:
filtered_tracks.head()

,mu_x,mu_y,mu_z,track_id,track_length,track_start,track_end,c3_peak,c2_peak,c1_peak,...,C3_adjusted_voxel_sum_peak_frame,C2_adjusted_voxel_sum_peak_frame,C1_adjusted_voxel_sum_peak_frame,max_z_movement,max_y_movement,max_x_movement,channel2_positive,channel1_positive,membrane_region,track_radius
1,105 124.0 301 124.0 424 123.0 655 ...,105 118.0 301 118.0 424 118.0 655 ...,105 45.0 301 44.0 424 46.0 655 ...,1,43,0,42,254.977143,201.971429,357.708571,...,32,37,34,5.0,2.0,6.0,True,True,Lateral,107.396462
2,32 30.0 282 30.0 352 31.0 628 ...,32 218.0 282 218.0 352 217.0 628 ...,32 6.0 282 8.0 352 6.0 628 ...,2,49,0,50,202.091429,171.640000,232.417143,...,5,3,9,5.0,6.0,6.0,False,False,Lateral,161.749807
3,8 51.0 306 50.0 447 51.0 667 ...,8 223.0 306 223.0 447 222.0 667 ...,8 22.0 306 22.0 447 22.0 667 ...,3,30,0,29,221.491429,186.382857,236.565714,...,26,24,20,5.0,1.0,3.0,True,False,Lateral,136.165341
7,58 112.0 177 112.0 508 113.0 576 ...,58 248.0 177 248.0 508 248.0 576 ...,58 41.0 177 42.0 508 40.0 576 ...,7,27,0,26,204.405714,173.211429,269.942857,...,18,24,18,7.0,2.0,6.0,False,False,Lateral,85.871998
9,75 18.0 164 19.0 481 20.0 522 ...,75 236.0 164 236.0 481 236.0 522 ...,75 8.0 164 7.0 481 7.0 522 8....,9,23,0,22,291.405714,273.491429,331.742857,...,8,6,8,2.0,1.0,3.0,True,True,Lateral,173.352243


In [ ]:
# value = filtered_tracks.iloc[1,0]
# print(value)
#track_df.head()
#print(filtered_tracks.columns)
print(filtered_tracks['membrane_region'])


1       False
2       False
3       False
7       False
9       False
        ...  
1733    False
1738    False
1758    False
1759    False
1774     True
Name: membrane_region, Length: 252, dtype: bool
